# 17 — Your first real Appium tool

We already have a Python function that captures the app's UI. In this lesson, we make it a LangChain tool and call it ourselves.

Follow the same pattern as `3_multi_agent_workflow.ipynb`: **import → define a tool → invoke it → read the result**.

```text
observe_current_ui.invoke({})
        ↓
observe_session_ui(session)
        ↓
Appium → XCUITest → running iOS app
        ↓
Receipt with screenshot and UI hierarchy paths
```

This is one manual observation of the normally configured app. The agent loop comes in a later lesson.


## 1) Imports

Appium setup and evidence capture are already implemented in the shared Python files. Here, import their functions and LangChain's `tool` decorator.

The small path setup lets the notebook run from the project root, `server/`, or `sample_scenarios/`.


In [1]:
import sys
from importlib import reload
from pathlib import Path
from langchain.tools import tool

# Find the shared modules folder from any supported notebook working directory.
folders = [Path.cwd(), Path.cwd() / "sample_scenarios", Path.cwd() / "server" / "sample_scenarios"]
for folder in folders:
    if (folder / "modules" / "appium_runtime.py").is_file():
        sys.path.insert(0, str(folder))
        break
else:
    raise RuntimeError("Open the kernel in the project root, server/, or sample_scenarios/.")

# Refresh the shared files so this kernel sees functions added since its last import.
from modules import appium_runtime, observations
reload(appium_runtime)
reload(observations)

from modules.appium_runtime import prepare_local_appium, start_ios_session, stop_ios_session
from modules.observations import observe_session_ui


## 2) Select the app

The bundle ID identifies Movies; the UDID identifies the Simulator. Keep this Simulator booted with the app installed, and run device lessons one at a time.

`prepare_local_appium()` selects and checks our existing local tools. It does not start a device session.


In [2]:
run_profile = {
    "id": "movies-ios-local",
    "platform": "ios",
    "execution_provider": "local",
    "app_bundle_id": "com.example.movies",
    "simulator_udid": "4F0707AE-45BC-464D-9099-6EE76C3740A9",
}

environment = prepare_local_appium()
print("Appium version:", environment.appium_version)


Appium version: 3.9.0


## 3) Define the tool

`@tool` turns an ordinary Python function into a LangChain tool. Its docstring describes what it does, just like the tools in your multi-agent notebook.

The function below calls our shared capture helper. `session` will be created in the next cell; Python reads that variable when the tool is invoked. Defining the tool here does not contact Appium.

This notebook uses one session at a time. The helper checks that the session is active and that the driver still has the expected session ID. Multi-user runtime binding is a later step.


In [3]:
@tool
def observe_current_ui() -> dict:
    """Capture the running app's current screen and native UI hierarchy."""
    return observe_session_ui(session)


## 4) Call the tool

Read these operations in order:

1. `start_ios_session(...)` starts our local Appium server and connects to the app.
2. `observe_current_ui.invoke({})` calls our tool. `{}` means it needs no public arguments.
3. `stop_ios_session(...)` closes the session and its owned server.

`finally` makes the stop call run even if observation raises an error. All resource-management details stay in [appium_runtime.py](modules/appium_runtime.py).


In [4]:
result = None  # Avoid showing an older result if this run fails.
session = start_ios_session(environment, run_profile)
try:
    result = observe_current_ui.invoke({})
finally:
    stop_ios_session(session)


Server/session log: /Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/logs/ios-session-6u0hreo5.log
Owned Appium PID: 48389
Automation session closed.
Owned Appium process stopped; exit code: 0


## 5) Read the result

For this argument-only `.invoke({})` call, the tool returns a Python dictionary: our **capture receipt**.

- `status`: `complete`, `partial`, or `failed`.
- `receipt_persisted`: whether the receipt was saved successfully.
- `errors`: any capture or recording problems.
- `artifacts`: paths to the captured hierarchy and screenshot, with validity flags.

A complete capture means we obtained evidence. It does not mean a movie scenario passed or that we verified a fresh backend request.


In [5]:
# A failed or skipped run has no receipt to inspect yet.
if globals().get("result") is None:
    print("No observation result yet. Run Step 4 successfully first; check its error if it fails.")
else:
    print("Capture status:", result["status"])
    print("Receipt saved:", result["receipt_persisted"])
    print("Errors:", result["errors"])
    print("Artifacts:", result["artifacts"])


Capture status: complete
Receipt saved: True
Errors: []
Artifacts: {'hierarchy': {'path': '/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/observations/observe-g0zmruwi/hierarchy.xml', 'started_at': '2026-11-03T15:42:29.463097+00:00', 'completed_at': '2026-11-03T15:42:29.688353+00:00', 'valid': True}, 'screenshot': {'path': '/Users/tarunkhurana/Engineering/projects/tarunkhurana2015/ai-certification-gen-academy/week3-project-automated-testing/server/.tools/observations/observe-g0zmruwi/screen.png', 'started_at': '2026-11-03T15:42:29.691814+00:00', 'completed_at': '2026-11-03T15:42:29.842766+00:00', 'valid': True}}


## What we learned

The essential LangChain pattern is just:

```python
@tool
def my_tool() -> dict:
    """Describe the operation."""
    return existing_python_function()

result = my_tool.invoke({})
```

Our real tool uses Appium through [observations.py](modules/observations.py). The shared helpers keep the device setup and cleanup out of the lesson's core code. The captured paths refer to files; they do not automatically send image/XML contents to a model.

Run the import cell again after editing the shared Python modules; it reloads their current definitions. Do this between completed runs, after the session is stopped. Then run the remaining cells top to bottom. The local iOS helper is the current execution adapter; Android, web, and HeadSpin remain later work.

**Verified:** `2026-11-03T15:42:29.871057+00:00` (UTC). The real iOS capture returned `complete`, with a saved receipt and no capture errors. The session and owned Appium server closed successfully.

[Captured screenshot](../.tools/observations/observe-g0zmruwi/screen.png) · [Capture receipt](../.tools/observations/observe-g0zmruwi/manifest.json)

[Index](README.md) · [Previous lesson](16_reusable_ui_operations.ipynb)
